# Hands-on 2: Urban Thermal Comfort Modeling

**Module 2 — Deep Learning Best Practices & Applications**  
CIMA Foundation PhD Course, 27 May 2026

In this notebook we build a neural network to predict a thermal comfort index (UTCI) from meteorological and urban variables.  
Same stack as Notebook 1: Zarr + xarray, PyTorch Lightning, OmegaConf, MLFlow.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/02_urban_thermal_comfort.ipynb)

## 0. Setup

Same setup as Notebook 1 — we use [**uv**](https://docs.astral.sh/uv/) to install dependencies. On Colab we install into the system Python with `uv pip install --system`. For local development, use `uv sync` to create a managed `.venv`.

In [ ]:
import os

# On Colab: clone the repo and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse 2>/dev/null || true
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import zarr
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from omegaconf import OmegaConf
import mlflow

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Configuration

In [ ]:
cfg = OmegaConf.create({
    "data": {
        "zarr_url": "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
        "cities": {
            "Rome":   {"lat": 41.9, "lon": 12.5},
            "Milan":  {"lat": 45.5, "lon": 9.2},
            "Berlin": {"lat": 52.5, "lon": 13.4},
        },
        "time_start": "2020-06-01",
        "time_end": "2020-08-31",
        "train_split": 0.8,
        "batch_size": 64,
    },
    "model": {
        "hidden_sizes": [64, 32, 16],
        "dropout": 0.1,
    },
    "training": {
        "learning_rate": 1e-3,
        "max_epochs": 30,
        "accelerator": "auto",
    },
    "mlflow": {
        "experiment_name": "urban_thermal_comfort",
    },
})

print(OmegaConf.to_yaml(cfg))

## 2. Data: ERA5 Urban Climate Variables

We load **real ERA5 reanalysis data** from the [ARCO-ERA5](https://github.com/google-research/arco-era5) cloud Zarr store (no download needed!).  
The store contains the full ERA5 archive at 0.25° resolution, hosted on Google Cloud Storage.

We extract surface variables for three European cities (Rome, Milan, Berlin) during summer 2020 and compute a simplified **UTCI (Universal Thermal Climate Index)** as the prediction target.

> **Note:** Opening the ARCO-ERA5 metadata may take ~30–60 seconds on first access.

In [ ]:
# Open the ARCO-ERA5 Zarr store (lazy — only metadata is read)
print("Opening ARCO-ERA5 store (this may take ~30-60s on first access)...")
ds_era5 = xr.open_zarr(
    cfg.data.zarr_url,
    chunks=None,
    storage_options=dict(token="anon"),
)
print("Store opened. Available variables:", len(ds_era5.data_vars))
print(ds_era5)

In [ ]:
# Extract surface variables for each city and build a tabular dataset
surface_vars = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "surface_solar_radiation_downwards",
    "mean_sea_level_pressure",
]

records = []
for city, coords in cfg.data.cities.items():
    print(f"Loading {city} ({coords.lat}°N, {coords.lon}°E)...")
    ds_city = (
        ds_era5[surface_vars]
        .sel(
            latitude=coords.lat,
            longitude=coords.lon,
            method="nearest",
        )
        .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
        .compute()
    )
    df = ds_city.to_dataframe().reset_index()
    df["city"] = city
    records.append(df)

df_all = pd.concat(records, ignore_index=True).dropna()
print(f"\nTotal records: {len(df_all)}")

# Derive features
df_all["t2m_C"] = df_all["2m_temperature"] - 273.15           # K → °C
df_all["d2m_C"] = df_all["2m_dewpoint_temperature"] - 273.15  # K → °C
df_all["wind_speed"] = np.sqrt(
    df_all["10m_u_component_of_wind"]**2 + df_all["10m_v_component_of_wind"]**2
)
df_all["solar_rad"] = df_all["surface_solar_radiation_downwards"].clip(lower=0)
df_all["mslp_hPa"] = df_all["mean_sea_level_pressure"] / 100  # Pa → hPa

# Simplified UTCI approximation (non-linear combination of meteorological variables)
humidity_effect = 0.5 * (df_all["t2m_C"] - df_all["d2m_C"])
df_all["utci"] = (
    0.7 * df_all["t2m_C"]
    + 0.2 * df_all["solar_rad"] / 100
    - 0.5 * df_all["wind_speed"]
    - 0.3 * humidity_effect
    + 0.01 * df_all["t2m_C"] * df_all["solar_rad"] / 100  # interaction
).astype(np.float32)

feature_names = ["t2m_C", "d2m_C", "wind_speed", "solar_rad", "mslp_hPa"]
features = df_all[feature_names].values.astype(np.float32)
utci = df_all["utci"].values.astype(np.float32)

print(f"Features shape: {features.shape}")
print(f"Target (UTCI) shape: {utci.shape}")
print(f"UTCI range: [{utci.min():.1f}, {utci.max():.1f}] °C")

# Show as xarray Dataset for exploration
ds = xr.Dataset({
    name: ("sample", features[:, i]) for i, name in enumerate(feature_names)
} | {"utci": ("sample", utci)})
print(ds)

In [ ]:
# Explore feature distributions and correlations
n_feat = len(feature_names)
ncols = 3
nrows = (n_feat + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
for i, (name, ax) in enumerate(zip(feature_names, axes.flat)):
    scatter = ax.scatter(features[:, i], utci, alpha=0.1, s=5, c=utci, cmap="RdYlBu_r")
    ax.set_xlabel(name)
    ax.set_ylabel("UTCI (°C)")
    ax.set_title(f"{name} vs UTCI")

# Hide unused subplot(s)
for j in range(n_feat, nrows * ncols):
    axes.flat[j].set_visible(False)

plt.suptitle("Feature-Target Relationships", fontsize=14)
plt.tight_layout()
plt.show()

## 3. PyTorch Dataset & DataModule

In [ ]:
class UrbanClimateDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.from_numpy(features)
        self.targets = torch.from_numpy(targets).unsqueeze(1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


class UrbanClimateDataModule(pl.LightningDataModule):
    def __init__(self, features, targets, cfg):
        super().__init__()
        self.features = features
        self.targets = targets
        self.cfg = cfg
        # Compute normalization stats on full data
        self.feat_mean = features.mean(axis=0)
        self.feat_std = features.std(axis=0)
        self.target_mean = targets.mean()
        self.target_std = targets.std()

    def setup(self, stage=None):
        # Normalize
        feat_norm = (self.features - self.feat_mean) / self.feat_std
        tgt_norm = (self.targets - self.target_mean) / self.target_std

        n = len(feat_norm)
        n_train = int(n * self.cfg.data.train_split)
        self.train_ds = UrbanClimateDataset(feat_norm[:n_train], tgt_norm[:n_train])
        self.val_ds = UrbanClimateDataset(feat_norm[n_train:], tgt_norm[n_train:])

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.cfg.data.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.cfg.data.batch_size)


dm = UrbanClimateDataModule(features, utci, cfg)
dm.setup()
print(f"Train: {len(dm.train_ds)}, Val: {len(dm.val_ds)}")

## 4. Model: MLP for Thermal Comfort Prediction

In [ ]:
class ThermalComfortMLP(pl.LightningModule):
    """MLP for predicting UTCI from meteorological + urban features."""

    def __init__(self, cfg, n_features):
        super().__init__()
        self.save_hyperparameters()
        self.cfg = cfg

        layers = []
        in_size = n_features
        for hidden_size in cfg.model.hidden_sizes:
            layers.extend([
                nn.Linear(in_size, hidden_size),
                nn.ReLU(),
                nn.Dropout(cfg.model.dropout),
            ])
            in_size = hidden_size
        layers.append(nn.Linear(in_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = F.mse_loss(pred, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = F.mse_loss(pred, y)
        mae = F.l1_loss(pred, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.cfg.training.learning_rate)


model = ThermalComfortMLP(cfg, n_features=len(feature_names))
print(model)

## 5. Training with Lightning + MLFlow

In [ ]:
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="mlp_thermal_comfort_v1"):
    mlflow.log_params(OmegaConf.to_container(cfg, resolve=True))

    trainer = pl.Trainer(
        max_epochs=cfg.training.max_epochs,
        accelerator=cfg.training.accelerator,
        enable_progress_bar=True,
        log_every_n_steps=5,
    )

    trainer.fit(model, dm)

    val_loss = trainer.callback_metrics.get("val_loss")
    val_mae = trainer.callback_metrics.get("val_mae")
    if val_loss is not None:
        mlflow.log_metric("final_val_loss", val_loss.item())
    if val_mae is not None:
        mlflow.log_metric("final_val_mae", val_mae.item())

    print(f"\nFinal val_loss: {val_loss}")
    print(f"Final val_mae: {val_mae}")

## 6. Evaluation & Visualization

In [ ]:
# Predictions on validation set
model.eval()
val_features = dm.val_ds.features
val_targets = dm.val_ds.targets

with torch.no_grad():
    val_pred = model(val_features)

# Denormalize back to UTCI scale
val_pred_utci = val_pred.numpy().squeeze() * dm.target_std + dm.target_mean
val_true_utci = val_targets.numpy().squeeze() * dm.target_std + dm.target_mean

In [ ]:
# Scatter plot: predicted vs true UTCI
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Prediction scatter
axes[0].scatter(val_true_utci, val_pred_utci, alpha=0.2, s=5)
lims = [min(val_true_utci.min(), val_pred_utci.min()),
        max(val_true_utci.max(), val_pred_utci.max())]
axes[0].plot(lims, lims, "r--", lw=2, label="Perfect prediction")
axes[0].set_xlabel("True UTCI (°C)")
axes[0].set_ylabel("Predicted UTCI (°C)")
axes[0].set_title("Predicted vs True UTCI")
axes[0].legend()

# Error distribution
errors = val_pred_utci - val_true_utci
axes[1].hist(errors, bins=50, edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="r", linestyle="--")
axes[1].set_xlabel("Prediction Error (°C)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Error Distribution (MAE={np.mean(np.abs(errors)):.2f}°C)")

plt.tight_layout()
plt.show()

# Print metrics
rmse = np.sqrt(np.mean(errors**2))
mae = np.mean(np.abs(errors))
r2 = 1 - np.sum(errors**2) / np.sum((val_true_utci - val_true_utci.mean())**2)
print(f"RMSE: {rmse:.2f} °C")
print(f"MAE:  {mae:.2f} °C")
print(f"R²:   {r2:.4f}")

In [ ]:
# Feature importance via permutation
baseline_mse = np.mean((val_pred_utci - val_true_utci) ** 2)
importance = {}

for i, name in enumerate(feature_names):
    feat_permuted = val_features.clone()
    feat_permuted[:, i] = feat_permuted[torch.randperm(len(feat_permuted)), i]
    with torch.no_grad():
        pred_permuted = model(feat_permuted).numpy().squeeze() * dm.target_std + dm.target_mean
    mse_permuted = np.mean((pred_permuted - val_true_utci) ** 2)
    importance[name] = mse_permuted / baseline_mse

# Plot
names = list(importance.keys())
values = list(importance.values())
sorted_idx = np.argsort(values)[::-1]

plt.figure(figsize=(8, 4))
plt.barh([names[i] for i in sorted_idx], [values[i] for i in sorted_idx])
plt.axvline(1.0, color="r", linestyle="--", label="Baseline")
plt.xlabel("MSE ratio (higher = more important)")
plt.title("Permutation Feature Importance")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Exercises

1. **Thermal comfort classes**: Convert UTCI to stress categories (no stress, moderate heat stress, strong heat stress) and train a classifier
2. **Architecture search**: Try different hidden layer sizes and depths — use Hydra multirun or manual sweeps
3. **Add batch normalization**: Insert `nn.BatchNorm1d` layers and compare performance
4. **More cities**: Add more European cities to `cfg.data.cities` and retrain — does the model generalize?
5. **More variables**: Load additional ERA5 variables from the ARCO-ERA5 store (e.g., `boundary_layer_height`, `soil_temperature_level_1`)
6. **Temporal features**: Add hour-of-day and day-of-year as cyclic features (sin/cos encoding)